In [ ]:
%matplotlib inline
import pandas as pd
import sionna.rt as rt
from pathlib import Path 
from utils.scene_utils import add_txs, add_rxs, set_frequence
from utils.material_factory import create_sionna_material

In [ ]:
SCENE_FILENAME = Path('data/blender/massy.xml')
TERRAIN_FILENAME = Path('data/blender/terrain.xml')
FREQUENCY = 3500  # Only supports: 700, 800, 900, 1800, 2100, 2600, 3500
FREQUENCY_UNIT='mhz' # Only supports: mhz, ghz
TRANSMITTER_DIRECTORY = Path('data/transimitters')
RECEIVER_FILENAME = Path('data/sensor_location.csv')

In [ ]:
# Load the Mitsuba description scene exported from Blender
scene = rt.load_scene(filename=SCENE_FILENAME)

set_frequence(scene, FREQUENCY, FREQUENCY_UNIT)

# Configure antenna arrays for transmitters (Tx) and receivers (Rx)
tx_array1 = rt.PlanarArray(
    num_rows=2,
    num_cols=2,
    vertical_spacing=0.5,
    horizontal_spacing=0.5,
    pattern="tr38901",
    polarization="cross",
    polarization_model="tr38901_2")
tx_array2 = rt.PlanarArray(
    num_rows=1,
    num_cols=1,
    vertical_spacing=0.5,
    horizontal_spacing=0.5,
    pattern="tr38901",
    polarization="cross")
rx_array = rt.PlanarArray(
    num_rows=1,
    num_cols=1,
    vertical_spacing=0.5,
    horizontal_spacing=0.5,
    pattern="dipole",
    polarization="cross")

scene.tx_array = tx_array1 if FREQUENCY == 3500 and FREQUENCY_UNIT=='mhz' else tx_array2
scene.rx_array = rx_array

In [ ]:
# Optional: Modify the radio material of some objects
scene.objects["water"].radio_material = create_sionna_material("freshwater", scene.frequency)
scene.objects["roads"].radio_material = create_sionna_material("asphalt_concrete", scene.frequency)
# To avoid confusion, discard the radio material initially loaded with the scene
scene.remove("wet_ground")
scene.remove("chipboard")

In [ ]:
# Load the preprocessed transmitter and receiver information from files
transmitter_filename = TRANSMITTER_DIRECTORY / f'{FREQUENCY}_{FREQUENCY_UNIT}.csv'

df_tx = pd.read_csv(transmitter_filename, encoding='utf-8-sig')
df_rx = pd.read_csv(RECEIVER_FILENAME, encoding='utf-8')

add_txs(scene, df_tx, TERRAIN_FILENAME)
add_rxs(scene, df_rx, TERRAIN_FILENAME)

In [ ]:
# Approach 1: Compute Radio Map on a flat bounding grid solver
rm_solver = rt.RadioMapSolver()

rm = rm_solver(scene,
    max_depth=5,           # Maximum number of ray-scene interactions
    samples_per_tx=10**7 , # Total Monte Carlo ray samples per Tx (higher = less noise, more memory)
    cell_size=(5, 5),      # Spatial resolution of each radio map grid cell (in meters)
    center=[0, 0, 0],      # Coordinate center point of the target radio map grid
    size=[4200, 3000],     # Total dimensions (x, y, in meters) of the radio map area
    orientation=[0, 0, 0]) # Spatial orientation of the evaluation grid planes


In [ ]:
# Visualize the calculated propagation metrics
rm.show(metric="path_gain")

# Visualize the Received Signal Strength (RSS)
rm.show(metric="rss")

# Visualize the Signal-to-Interference-plus-Noise Ratio (SINR)
rm.show(metric="sinr")

In [ ]:
# Approach 2: Compute Radio Map adaptively conforming to the continuous 3D terrain surface mesh
rm_solver = rt.RadioMapSolver()

# Extract and duplicate the terrain surface as the continuous measuring mesh target
mesurement_surface = scene.objects["terrain"].clone(as_mesh=True)

rm = rm_solver(scene,
    measurement_surface=mesurement_surface,
    samples_per_tx=10**8,   # Increased ray counts for complex surface conformal calculations
    max_depth=5)

In [ ]:
# Render interactive previews of the 3D scene topology
scene.preview()

In [ ]:
# Preview the 3D scene overlaid with the computed Radio Map coverage
scene.preview(radio_map=rm, rm_vmin=-200)